# PIPELINE DE ANÁLISE SEMÂNTICA — PEEL PHASE 1

Este notebook implementa um pipeline de análise semântica e clustering lexical correspondente à Fase 1 do PEEL. Para isso, o sistema utiliza diferentes ferramentas de NLP, Word Sense Disambiguation (WSD), embeddings semânticos e clustering automático.

## Ferramentas utilizadas

- spaCy
- NLTK
- WordNet
- GlossBERT
- Sentence-BERT
- HDBSCAN
- PyTorch
- Transformers (Hugging Face)

---

# OBJETIVO

O sistema identifica stems frequentes e os sentidos semânticos mais prováveis das palavras derivadas desses stems em um corpus textual. Além disso, o pipeline:

- valida ambiguidades lexicais via Word Sense Disambiguation (WSD);
- gera definições semânticas contextualizadas;
- cria clusters semânticos concisos;
- permite revisão manual de definições e agrupamentos;
- exporta os resultados estruturados para JSON e HTML.

---

# REQUISITOS

## Arquivos necessários

- Corpus textual `.txt`
- Pasta local:

```text
./GlossBERT_Checkpoint
```

## A pasta `GlossBERT_Checkpoint` deve conter:

- `config.json`
- `pytorch_model.bin`
- `vocab.txt`
- `tokenizer_config.json`
- demais arquivos do checkpoint do modelo

---

# DEPENDÊNCIAS

```bash
pip install torch spacy nltk transformers sentence-transformers hdbscan numpy tqdm
```

---

# RECURSOS NLTK NECESSÁRIOS

```python
nltk.download("wordnet")
nltk.download("omw-1.4")
```

---

# RECURSO SPACY NECESSÁRIO

```python
python -m spacy download en_core_web_sm
```

---
# FLUXO GERAL

1. Processamento linguístico do corpus  
2. Extração de stems frequentes  
3. Mapeamento:
   - palavra
   - stem
   - POS
   - sentença
4. Recuperação de synsets WordNet  
5. Predição semântica com GlossBERT  
6. Revisão manual opcional  
7. Geração de definições aceitas  
8. Criação de embeddings semânticos (Sentence-BERT)  
9. Clustering automático (HDBSCAN)  
10. Revisão manual dos clusters  
11. Exportação final para JSON e HTML  

---

# OBSERVAÇÕES

- Alguns stems podem não existir no WordNet
- Clusters considerados ruído são descartados automaticamente
- Execução com GPU é recomendada para melhor desempenho
- O sistema permite revisão manual tanto das definições quanto dos clusters semânticos
- O pipeline foi projetado para apoiar workflows interpretativos do PEEL localmente

In [ ]:
# IMPORTS
from collections import Counter, defaultdict
import math
import torch
import spacy
import nltk
from nltk.stem import PorterStemmer
from tqdm.auto import tqdm
from transformers import (BertTokenizer, BertForSequenceClassification)
from nltk.corpus import wordnet as wn
from sentence_transformers import SentenceTransformer
import hdbscan
import numpy as np
import json
import re

# REQUIRED DOWNLOADS
nltk.download("wordnet")
nltk.download("omw-1.4")

# REQUIRED UPLOADS
# Upload the .txt file you wish to analyze and
# also the GlossBERT_Checkpoint folder
# Find GlossBERT_Checkpoint here: https://github.com/HSLCY/GlossBERT

In [ ]:
# DEVICE
# This pipeline requires a GPU or else it'll take a long time or not even run at all. Choose an environment with a GPU and check if the output here is "cuda"
# If output is "cuda", proceed

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"\nUsing device: {device}")

In [ ]:
# FUNCTION TO AUTOMATICALLY NAME A CORPUS:

def corpus_name(text):
  match = re.match(r"^[^.]+", text)
  if match:
    result = match.group(0)
    return(result)

In [ ]:
# GLOBAL VARIABLES AND MODELS

# VARIABLES
TXT_FILE = r"CHAP3ofMagnificaHumanitas.txt"      # Path to your .txt file, here I already included an example
CORPUS_NAME = corpus_name(TXT_FILE)
TOP_PERCENTILE = 0.50       # Example: top 50% most frequent stems
MAX_STEMS = 150             # Maximum number of stems to return
TOP_STEMS = "top_stems.txt" # File with top stems
GLOSSBERT_OUTPUT = "glossbert_accepted_terms.txt" # File with GlossBERT results
MAX_SENTENCES_PER_STEM = 5 # For each collected stems, get N sentences that contain it
MAX_SYNSETS = 5 # For each stem, get 5 WordNet definitions of terms represented by it
MAX_CLUSTER_SIZE = 10 # Maximum number of stems in a cluster for it to be considered a valid cluster when running clusters for the first time 
MIN_CLUSTERS = 5 # Minimum number of clusters to be generated by HDBSCAN, this is a parameter that can be tuned to get more or fewer clusters, but I found that 5 is a good number for small texts
MIN_CLUSTER_LEN = 3 # Minimum number of stems in a cluster for it to be considered a valid cluster when rerunning noisy clusters, this is a parameter that can be tuned to get more or fewer clusters, but I found that 3 is a good number for small texts
FINAL_JSON = f"{CORPUS_NAME}-phase1_state.json"
HTML_PATH = f"{CORPUS_NAME}-Phase1-clusters.html" # Tableau20 HTML
VOYANT_NOTEBOOK = f"PEEL-TemplateSN.html"
OUTPUT_NOTEBOOK = "voyant_phase1.html"



# MODELS
LANG_MODEL = "en_core_web_sm" # spaCy's language model
nlp = spacy.load(LANG_MODEL)
stemmer = PorterStemmer() # Stemmer from NLTK, spaCy does not work with stems (only lemmas)
MODEL_PATH = "./GlossBERT_Checkpoint" # GlossBERT to perform definition-checking steps, find it here: https://github.com/HSLCY/GlossBERT
sentence_embedder = "all-MiniLM-L6-v2" # Model to be used to generate sentence embeddings for semantic clustering

In [ ]:
# READ TEXT FILE

with open(TXT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

# PROCESS TEXT

doc = nlp(text)


# EXTRACT STEMS USING NLTK


tokens = []

for token in doc:
    if (
        not token.is_stop and      # remove stopwords
        not token.is_punct and     # remove punctuation
        not token.is_space and     # remove spaces
        token.is_alpha             # keep alphabetic tokens only
    ):
        stem = stemmer.stem(token.text.lower())
        tokens.append(stem)

# COUNT FREQUENCIES

freq = Counter(tokens)

# SORT BY FREQUENCY DESCENDING
sorted_freq = sorted(freq.items(), key=lambda x: x[1], reverse=True)

# APPLY PERCENTILE FILTER

percentile_n = math.ceil(len(sorted_freq) * TOP_PERCENTILE)

final_n = min(percentile_n, MAX_STEMS) # Respect both percentile and absolute maximum

top_stems = dict(sorted_freq[:final_n]) # Final selection

# OUTPUT

print(f"\nTotal unique stems: {len(sorted_freq)}")
print(f"Returning top {final_n} stems\n")

for stem, count in top_stems.items():
    print(f"{stem}: {count}")


# SAVE TO FILE OS HASH (#) IF UNDESIRED


with open(TOP_STEMS, "w", encoding="utf-8") as out:
    out.write("stem\tcount\n")
    for stem, count in top_stems.items():
        out.write(f"{stem}\t{count}\n")

print(f"\nSaved results to {TOP_STEMS}")

In [ ]:
# MAPPING OF SPACY'S POS CATEGORIES ONTO NLTK
# This code uses WordNet, downloaded via NLTK, for word definitions, but spaCy's POS tagger is more up-to-date
# Therefore, I map spaCy's tags onto NLTK for POS representations

POS_MAP = {

    "NOUN": wn.NOUN,

    "VERB": wn.VERB,

    "ADJ": wn.ADJ,

    "ADV": wn.ADV
}

# LOAD GLOSSBERT CHECKPOINT

print("\nLoading GlossBERT checkpoint...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_PATH
)

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)

model.eval()

print("GlossBERT loaded successfully.")

# EXTRACTION OF SENTENCES THAT CONTAIN THE RELEVANT STEMS

print("\nMapping stems to sentences...")

stem_occurrences = defaultdict(list)

sentences_list = list(doc.sents)

print("\nMapping stem occurrences...")

sentences_list = list(doc.sents)

for sent in tqdm(
    sentences_list,
    desc="Sentence mapping"
):

    sent_text = sent.text.strip()

    for token in sent:

        if not token.is_alpha:
            continue

        stem = stemmer.stem(token.text.lower())

        if stem not in top_stems:
            continue

        occurrence = {

            # Actual surface word
            "word": token.text,

            # Stem
            "stem": stem,

            # POS tag
            "pos": token.pos_,

            # Sentence
            "sentence": sent_text,

            # Character positions
            "start": token.idx,
            "end": token.idx + len(token.text)
        }

        stem_occurrences[stem].append(
            occurrence
        )
        #print(f"Mapped occurrence of stem '{stem}' in sentence: {sent_text}") # Unhash for visualization of stem occurrences
        #print(stem_occurrences[stem]) # Unhash for visualization of stem occurrences

# GLOSSBERT FUNCTION

def glossbert_predict(occurrence):

    word = occurrence["word"]

    pos = occurrence["pos"]

    sentence = occurrence["sentence"]

    wn_pos = POS_MAP.get(pos)

    # POS-filtered synsets using WORD

    synsets = wn.synsets(
        word,
        pos=wn_pos
    )

    if len(synsets) == 0:
        return None

    synsets = synsets[:MAX_SYNSETS]

    results = []

    # Mark word for classification according to GlossBERT's paper

    marked_sentence = sentence.replace(
        word,
        f"""" "{word}" """,
        1
    )

    for syn in synsets:

        gloss = syn.definition()

        encoding = tokenizer(
            marked_sentence,
            gloss,
            return_tensors="pt",
            truncation=True,
            max_length=128, # This can be increased so that more context is taken into account, but it will also increase computational demand without much benefit (I briefly tested it), so I set it to 128 as a default
            padding="max_length"
        )

        input_ids = encoding[
            "input_ids"
        ].to(device)

        attention_mask = encoding[
            "attention_mask"
        ].to(device)

        with torch.no_grad():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            probs = torch.softmax(
                logits,
                dim=1
            )

            match_score = probs[
                0
            ][1].item()

        results.append({

            "synset": syn,

            "definition": gloss,

            "score": match_score
        })

    results = sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )

    return results

# GLOSSBERT RESULTS

print("\nRunning GlossBERT analysis...")

accepted_definitions = {}

flagged_words = []

print("\nRunning GlossBERT analysis...")

accepted_definitions = {}

flagged_words = []

for stem, count in tqdm(
    top_stems.items(),
    total=len(top_stems),
    desc="GlossBERT"
):

    occurrences = stem_occurrences[stem]

    if len(occurrences) == 0:
        continue

    occurrences = occurrences[
        :MAX_SENTENCES_PER_STEM
    ] # Limit max sentences per stem for saving computational demand

    mismatch_found = False
    for occurrence in occurrences:

        word = occurrence[
            "word"
        ].lower()

        pos = occurrence["pos"]

        wn_pos = POS_MAP.get(pos)

        # WORD-LEVEL SYNSETS


        synsets = wn.synsets(
            word,
            pos=wn_pos
        )

        # SKIP IF NO SYNSETS

        if len(synsets) == 0:
            continue

        synsets = synsets[
            :MAX_SYNSETS
        ]

        # DEFAULT SENSE FOR WORD

        default_sense = synsets[0]

        # GLOSSBERT PREDICTION

        results = glossbert_predict(
            occurrence
        )

        if results is None:
            continue

        best_sense = results[0]["synset"]

        # FLAG MISMATCHES

        if (
            best_sense.name()
            != default_sense.name()
        ):

            mismatch_found = True
            #TODO think about other ways of computing mismatches if mismatches list is constantly large

            flagged_words.append({

                "word":
                    occurrence["word"],
                "stem":
                    stem,
                "pos":
                    pos,
                "count":
                    count,
                "sentence":
                    occurrence["sentence"],
                "default_sense":
                    default_sense.name(),
                "default_definition":
                    default_sense.definition(),
                "predicted_sense":
                    best_sense.name(),
                "predicted_definition":
                    best_sense.definition(),
                "top_candidates": [

                    {
                        "sense":
                            r["synset"].name(),

                        "definition":
                            r["definition"],

                        "score":
                            round(
                                r["score"],
                                4
                            )
                    }

                    for r in results[:3]
                ]
            })
            #break # Hash if you want to verify every single mismatch occurrence, but they may compose a large amount of cases for a single stem
            # If you keep 'break' unhashed, the code will stop at the first mismatch occurrence of each stem, which can be useful to verify if the mismatches are valid or if they are just noise, but it may also hide a large amount of mismatches for stems that have a lot of occurrences, so it's up to you to decide whether to keep it or not based on your needs and the size of your dataset
            ################## IMPORTANT BREAK HERE ##################

    # ACCEPT DEFAULT SENSE

    if not mismatch_found:

        for occurrence in occurrences:

            word = occurrence[
                "word"
            ].lower()

            pos = occurrence["pos"]

            wn_pos = POS_MAP.get(pos)

            synsets = wn.synsets(
                word,
                pos=wn_pos
            )

            if len(synsets) == 0:
                continue

            if stem not in accepted_definitions:
                accepted_definitions[stem] = {

                    "words":
                        set(),

                    "pos":
                        set(),

                    "count":
                        count,

                    "instances": []
                }

                accepted_definitions[stem]["words"].add(word)
                accepted_definitions[stem]["pos"].add(pos)
                accepted_definitions[stem]["instances"].append({
                    "word": word,
                    "definition": synsets[0].definition(),
                    "sentence": occurrence["sentence"]
                })

            else:

                accepted_definitions[stem]["words"].add(word)
                accepted_definitions[stem]["pos"].add(pos)
                accepted_definitions[stem]["instances"].append({
                    "word": word,
                    "definition": synsets[0].definition(),
                    "sentence": occurrence["sentence"]
                })
        
# OUTPUT RESULTS

print("\n===================================")
print("ACCEPTED DEFINITIONS")
print("===================================\n")

for stem, data in accepted_definitions.items():

    print({
        "words": sorted(list(data["words"])),
        "stem": stem,
        "pos": sorted(list(data["pos"])),
        # "count": data["count"], # This is the stem overall count, which is not very relevant at this stage, so I hide it for better readability, but it can be uncommented if desired
        "instances": data["instances"]})

# FLAGGED TERMS

print("\n===================================")
print("FLAGGED TERMS")
print("===================================\n")

for item in flagged_words:

    print("\n-----------------------------------")

    print(f"WORD: {item['word']}")

    print(f"STEM: {item['stem']}")

    print(f"POS: {item['pos']}")

    # print(f"COUNT: {item['count']}") # This is the stem overall count, which is not very relevant at this stage, so I hide it for better readability, but it can be uncommented if desired

    print("\nSENTENCE:")

    print(item["sentence"])

    print("\nDEFAULT SENSE:")

    print(item["default_sense"])

    print(item["default_definition"])

    print("\nPREDICTED SENSE:")

    print(item["predicted_sense"])

    print(item["predicted_definition"])

    print("\nTOP 3 CANDIDATES:")

    for candidate in item["top_candidates"]:

        print(
            f"- {candidate['sense']}"
        )

        print(
            f"  DEF: "
            f"{candidate['definition']}"
        )

        print(
            f"  SCORE: "
            f"{candidate['score']}"
        )


print("\nDone.")

In [ ]:
# USER REVIEW OF FLAGGED TERMS

print("\n===================================")
print("FLAGGED TERM REVIEW")
print("===================================\n")

if len(flagged_words) == 0:

    print("No flagged terms found.")

else:

    # ASK USER IF THEY WANT TO ACCEPT ALL

    accept_all = input(
        "\nAccept all predicted definitions "
        "from flagged terms? (y/n): "
    ).strip().lower()

    # ACCEPT ALL

    if accept_all == "y":

        for item in flagged_words:

            stem = item["stem"]

            if stem not in accepted_definitions:
                accepted_definitions[stem] = {

                    "words":
                        set(),

                    "pos":
                        set(),

                    "count":
                        item["count"],

                    "instances": [],

                }

                accepted_definitions[stem]["words"].add(item["word"])
                accepted_definitions[stem]["pos"].add(item["pos"])
                accepted_definitions[stem]["instances"].append({
                    "word": item["word"],
                    "definition": item["predicted_definition"],
                    "sentence": item["sentence"]
                })
            else:

                accepted_definitions[stem]["words"].add(item["word"])
                accepted_definitions[stem]["pos"].add(item["pos"])
                accepted_definitions[stem]["instances"].append({
                    "word": item["word"],
                    "definition": item["predicted_definition"],
                    "sentence": item["sentence"]
                })

        print(
            "\nAll predicted definitions accepted."
        )

    # MANUAL REVIEW

    else:

        review_mode = input(

            "\nReview flagged terms:\n"
            "[1] One by one\n"
            "[2] Select from all terms\n\n"
            "Choice: "

        ).strip()

        # FUNCTION TO REVIEW ONE ITEM

        def review_flagged_item(item):

            print("\n===================================")

            print(f"WORD: {item['word']}")

            print(f"STEM: {item['stem']}")

            print(f"POS: {item['pos']}")

            print(f"COUNT: {item['count']}")

            print("\nSENTENCE:")

            print(item["sentence"])

            print("\nDEFAULT SENSE:")

            print(
                f"{item['default_sense']}"
            )

            print(
                f"{item['default_definition']}"
            )

            print("\nPREDICTED SENSE:")

            print(
                f"{item['predicted_sense']}"
            )

            print(
                f"{item['predicted_definition']}"
            )

            print("\nTOP CANDIDATES:\n")


            # PRINT CANDIDATES FOR UPDATED DEFINITION

            for idx, candidate in enumerate(
                item["top_candidates"],
                start=1
            ):

                print(
                    f"[{idx}] "
                    f"{candidate['sense']}"
                )

                print(
                    f"DEF: "
                    f"{candidate['definition']}"
                )

                print(
                    f"SCORE: "
                    f"{candidate['score']}\n"
                )

            # USER CHOICE PART

            print(
                "[0] Keep default definition"
            )

            print(
                "[4] Enter manual definition"
            )

            print(
                "Or type 'Accept all' to accept all remaining flagged terms"
            )

            choice = input(
                "\nChoice: "
            ).strip()
            
                        
            # ACCEPT ALL REMAINING ITEMS

            if choice.lower() == "accept all":

                return "accept_all"


            # IF KEEP DEFAULT

            if choice == "0":

                chosen_definition = (
                    item["default_definition"]
                )

            # MANUAL DEFINITION

            elif choice == "4":

                chosen_definition = input(
                    "\nEnter custom definition: "
                ).strip()

            # CHOOSE FROM TOP CANDIDATES

            elif choice in ["1", "2", "3"]:

                candidate = item[
                    "top_candidates"
                ][int(choice) - 1]

                chosen_definition = (
                    candidate["definition"]
                )
            
            elif choice.lower() == "exit":
                
                return "exit"
                

            # IF INVALID INPUT

            else:

                print(
                    "\nInvalid option."
                )

                return

            # UPDATE ACCEPTED DEFINITIONS

            stem = item["stem"]

            if stem not in accepted_definitions:
                accepted_definitions[stem] = {

                    "words":
                        set(),

                    "pos":
                        set(),

                    "count":
                        item["count"],

                    "instances":
                        []
                }

                accepted_definitions[stem]["words"].add(item["word"])
                accepted_definitions[stem]["pos"].add(item["pos"])
                accepted_definitions[stem]["instances"].append({
                    "word": item["word"],
                    "definition": chosen_definition,
                    "sentence": item["sentence"]
                })
            else:

                accepted_definitions[stem]["words"].add(item["word"])
                accepted_definitions[stem]["pos"].add(item["pos"])
                accepted_definitions[stem]["instances"].append({
                    "word": item["word"],
                    "definition": chosen_definition,
                    "sentence": item["sentence"]
                })
            print(
                "\nDefinition updated."
            )

        # ONE-BY-ONE REVIEW

        if review_mode == "1":

            for idx, item in enumerate(flagged_words):
                result = review_flagged_item(item)

                # ACCEPT ALL REMAINING ITEMS

                if result == "accept_all":

                    remaining_items = flagged_words[idx:]

                    for remaining_item in remaining_items:

                        stem = remaining_item["stem"]

                        if stem not in accepted_definitions:

                            accepted_definitions[stem] = {
                                "words": set(),
                                "pos": set(),
                                "count": remaining_item["count"],
                                "instances": []
                            }

                        accepted_definitions[stem]["words"].add(remaining_item["word"])

                        accepted_definitions[stem]["pos"].add(remaining_item["pos"])

                        accepted_definitions[stem]["instances"].append({
                            "word": remaining_item["word"],
                            "definition": remaining_item["predicted_definition"],
                            "sentence": remaining_item["sentence"]
                        })

                    print("\nAll remaining flagged terms accepted.")

                    break
                
                elif result == "exit":

                    print("\nExiting review.")

                    break

        # SELECTIVE REVIEW

        elif review_mode == "2":

            print(
                "\nFLAGGED TERMS:\n"
            )

            for item in flagged_words:

                print(
                    f"- {item['word']} "
                    f"(stem={item['stem']})"
                )

            while True:

                selected_word = input(

                    "\nType a word to review "
                    "(or 'exit'): "

                ).strip()

                if selected_word.lower() == "exit":
                    break

                found = False

                for item in flagged_words:

                    if (
                        item["word"].lower()
                        == selected_word.lower()
                    ):

                        review_flagged_item(item)

                        found = True

                        

                if not found:

                    print(
                        "\nWord not found."
                    )

# SAVE UPDATED RESULTS

print("\nSaving updated results...")

with open(
    GLOSSBERT_OUTPUT,
    "w",
    encoding="utf-8"
) as out:

    out.write(
        "stem\twords\tcount\tpos\tdefinitions\n"
    )

    for stem, data in accepted_definitions.items():

        out.write(
            f"{stem}\t"
            f"{data['words']}\t"
            f"{data['count']}\t"
            f"{data['pos']}\t"
            f"{data['instances']}\n"
        )

print(
    f"\nUpdated results saved to "
    f"{GLOSSBERT_OUTPUT}"
)

In [ ]:
# Visualization of accepted definitions to check if everything is in order before proceeding to clusterizations
print("\n===================================")
print("FULL DICTIONARY OF ACCEPTED DEFINITIONS")
print("===================================\n")
print(accepted_definitions)
print("\n-----------------------------------\n")
print("STEM, DATA - ONE BY ONE REPRESENTATION\n")
print("\n-----------------------------------")
for stem, data in accepted_definitions.items():
    print(f"{stem}, {data}")

In [ ]:
# LOAD SENTENCE-BERT

embedder = SentenceTransformer(
    sentence_embedder
)

# BUILD TEXT REPRESENTATIONS BY VECTORIZING SENTENCES

stem_texts = []
stem_names = []

for stem, data in accepted_definitions.items():

    for instance in data["instances"]:
        text_representation = f"{instance['word']} means '{instance['definition']}' in sentence: '{instance['sentence']}'"
        stem_texts.append(
            text_representation
        )
        stem_names.append(stem)


# VISUALIZATION OF FIRST DATA POINT TO CHECK IF EVERYTHING IS IN ORDER
print("\n===================================")
print("SAMPLE STEM REPRESENTATION")
print("===================================\n")
print(f"stem: {stem_names[0]}")
print(f"definition: {stem_texts[0]}")
print("\n===================================\n")
# CREATE EMBEDDINGS

embeddings = embedder.encode(
    stem_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# HDBSCAN CLUSTERING

clusterer = hdbscan.HDBSCAN(

    min_cluster_size=MIN_CLUSTERS, # The default is 5, but I thought that it could lead to noise in small texts, so I set it to 3

) # Other params are left as default

labels = clusterer.fit_predict(
    embeddings
)

# ORGANIZE CLUSTERS

clusters = {}

for stem, label in zip(
    stem_names,
    labels
):
    definition = stem_texts[stem_names.index(stem)]
    print(f"STEM: {stem}\n Definition: {definition}\n Label: {label}\n") # Hash/Unhash for visualization of cluster assignments
    # Ignore noise (all -1 categories are non-confident clusters)

    if label == -1:
        continue

    if label not in clusters:
        clusters[label] = []

    if stem not in clusters[label]:
        clusters[label].append(
            stem
        )


# Check if dimensions match before printing clusters
print(len(stem_names))
print(len(stem_texts))
print(len(embeddings))
print(len(labels))
# CLUSTERS ARE PRINTED IN THE NEXT CELL

In [ ]:
# CLUSTER RENAMING
# This cell composes the cluster-labeling step. It's characterized by a simple naming process
# That chooses cluster titles by frequency

stem_word_frequencies = {}

for stem, occurrences in stem_occurrences.items():

    words = [

        occ["word"].lower()

        for occ in occurrences
    ]

    stem_word_frequencies[stem] = Counter(words)

# NAME CLUSTERS BASED ON FREQUENCY OF TOP-OCCURRING WORD(S)

renamed_clusters = {}

for label, stems in clusters.items():

    # Aggregate word frequencies

    cluster_word_counter = Counter()

    for stem in stems:

        if stem in stem_word_frequencies:

            cluster_word_counter.update(
                stem_word_frequencies[stem]
            )

    # Skip empty clusters

    if len(cluster_word_counter) == 0:
        continue

    # Find highest frequency

    max_freq = max(
        cluster_word_counter.values()
    )

    # Get all tied words

    top_words = sorted([

        word

        for word, freq in
        cluster_word_counter.items()

        if freq == max_freq
    ])

    # Build cluster name

    cluster_name = " & ".join(
        top_words
    ) # Words are joined if tied for highest frequency

    renamed_clusters[
        cluster_name
    ] = {

        "label": label,

        "stems": stems,

        "top_frequency": max_freq,

        "top_words": top_words
    }

# SEMANTIC CLUSTERS

print("\n======================")
print("SEMANTIC CLUSTERS")
print("======================\n")

for cluster_name in sorted(
    renamed_clusters.keys()
):

    cluster_data = renamed_clusters[
        cluster_name
    ]

    print(
        f"CLUSTER: {cluster_name}"
    )

    print(
        "STEMS:"
    )

    print(
        ", ".join(
            cluster_data["stems"]
        )
    )

    print()

In [ ]:
# =====================================================
# RECLUSTER LARGE CLUSTERS
# =====================================================

large_clusters = {

    label: stems

    for label, stems in clusters.items()

    if len(set(stems)) > MAX_CLUSTER_SIZE
}

# =====================================================
# MAP STEM -> ORIGINAL LARGE CLUSTER
# =====================================================

stem_to_parent_cluster = {}

for label, stems in large_clusters.items():

    for stem in stems:

        stem_to_parent_cluster[
            stem
        ] = label

# =====================================================
# COLLECT UNIQUE STEMS
# =====================================================

large_stems = set()

for stems in large_clusters.values():

    large_stems.update(
        stems
    )

# =====================================================
# BUILD ENRICHED REPRESENTATIONS
# =====================================================

large_texts = []
large_stem_names = []

for stem in large_stems:

    contexts = []
    candidate_definitions = []
    observed_words = []

    if stem not in stem_occurrences:
        continue

    for occurrence in stem_occurrences[stem]:

        observed_words.append(
            occurrence["word"]
        )

        contexts.append(
            occurrence["sentence"]
        )

        results = glossbert_predict(
            occurrence
        )

        if results is None:
            continue

        candidate_definitions.extend([

            candidate["definition"]

            for candidate in results[:3]

        ])

    # Remove duplicates

    observed_words = list(
        dict.fromkeys(
            observed_words
        )
    )

    contexts = list(
        dict.fromkeys(
            contexts
        )
    )

    candidate_definitions = list(
        dict.fromkeys(
            candidate_definitions
        )
    )

    # Limit length

    observed_words = observed_words[:5]

    contexts = contexts[:3]

    candidate_definitions = (
        candidate_definitions[:5]
    )

    text_representation = (

        f"Stem: {stem}. "

        f"Observed words: "

        + ", ".join(
            observed_words
        )

        + ". Contexts: "

        + " ".join(
            contexts
        )

        + ". Candidate senses: "

        + " ".join(
            candidate_definitions
        )
    )

    large_texts.append(
        text_representation
    )

    large_stem_names.append(
        stem
    )

# =====================================================
# CREATE EMBEDDINGS
# =====================================================

large_embeddings = embedder.encode(

    large_texts,

    convert_to_numpy=True,

    normalize_embeddings=True

)

# =====================================================
# RECLUSTER
# =====================================================

large_labels = hdbscan.HDBSCAN(

    min_cluster_size=MIN_CLUSTERS,

    min_samples=1,

    cluster_selection_method="leaf"

).fit_predict(

    large_embeddings
)

# =====================================================
# BUILD SUBCLUSTERS
# =====================================================

large_subclusters = {}

for stem, label in zip(

    large_stem_names,

    large_labels

):

    if label == -1:
        continue

    if label not in large_subclusters:

        large_subclusters[label] = set()

    large_subclusters[label].add(
        stem
    )

# =====================================================
# REMOVE SUBCLUSTERS BELOW THRESHOLD
# =====================================================

large_subclusters = {

    label: stems

    for label, stems in large_subclusters.items()

    if len(stems) >= MIN_CLUSTER_LEN
}

print(
    f"\nGenerated "
    f"{len(large_subclusters)} "
    f"valid subclusters."
)

# =====================================================
# DETERMINE WHICH PARENT CLUSTERS SPLIT
# =====================================================

parent_to_subclusters = {}

for subcluster_label, stems in (

    large_subclusters.items()

):

    parent_labels = {

        stem_to_parent_cluster[stem]

        for stem in stems

        if stem in stem_to_parent_cluster
    }

    for parent_label in parent_labels:

        if parent_label not in parent_to_subclusters:

            parent_to_subclusters[
                parent_label
            ] = []

        parent_to_subclusters[
            parent_label
        ].append(
            subcluster_label
        )

# =====================================================
# REPLACE ORIGINAL LARGE CLUSTERS
# =====================================================

next_cluster_label = (

    max(clusters.keys()) + 1

    if len(clusters) > 0

    else 0
)

for parent_label, subcluster_labels in (

    parent_to_subclusters.items()

):

    # Only replace if the parent actually split

    if len(subcluster_labels) <= 1:

        continue

    print(
        f"\nReplacing large cluster "
        f"{parent_label} "
        f"with "
        f"{len(subcluster_labels)} "
        f"subclusters."
    )

    # Remove original cluster

    if parent_label in clusters:

        del clusters[parent_label]

    # Add new subclusters

    for subcluster_label in subcluster_labels:

        clusters[
            next_cluster_label
        ] = list(

            sorted(

                large_subclusters[
                    subcluster_label
                ]
            )
        )

        next_cluster_label += 1

# =====================================================
# PRINT UPDATED CLUSTERS
# =====================================================

print("\n======================")
print("UPDATED CLUSTERS")
print("======================\n")

for label, stems in sorted(
    clusters.items()
):

    print(
        f"CLUSTER {label}"
    )

    print(
        ", ".join(
            sorted(
                set(stems)
            )
        )
    )

    print()

In [ ]:
# =====================================================
# PRINT RESULTS FROM SUBCLUSTERS ONLY
# =====================================================

print("\n======================")
print("LARGE CLUSTER SUBCLUSTERS")
print("======================\n")

for label, stems in large_subclusters.items():

    print(
        f"SUBCLUSTER {label}"
    )

    print(
        ", ".join(
            sorted(stems)
        )
    )

    print()

In [ ]:
# Extract unique stems that were classified as noise

noise_stems = sorted(set(
    stem
    for stem, label in zip(stem_names, labels)
    if label == -1
))

noise_texts = []
noise_stem_names = []

for stem in noise_stems:

    for occurrence in stem_occurrences[stem]:

        results = glossbert_predict(
            occurrence
        )

        if results is None:
            continue

        word = occurrence["word"]

        sentence = occurrence["sentence"]

        text_representation = (
            f"Word: {word}. "
            f"Sentence: {sentence}. "
            f"Candidate senses: "
            + " ".join(
                candidate["definition"]
                for candidate in results[:3]
            )
        )

        noise_texts.append(
            text_representation
        )

        noise_stem_names.append(
            stem
        )

# Create embeddings

noise_embeddings = embedder.encode(
    noise_texts,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Cluster only the former noise points

noise_labels = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTERS
).fit_predict(
    noise_embeddings
)

In [ ]:
# Print new cluster based on rerunning previous noisy clusters
# Clusters were identified by HDBSCAN as non-confident, but they may still contain some relevant stems that were not clustered in the first round, so I rerun clustering on them (using top synsets GlossBERT predicted) to see if I can recover some of those stems in new clusters, which are printed here as "NOISE CLUSTERS"

noise_clusters = {}

for stem, label in zip(
    noise_stem_names,
    noise_labels
):

    if label == -1:
        continue

    if label not in noise_clusters:
        noise_clusters[label] = []

    noise_clusters[label].append(stem)

# Remove small clusters

noise_clusters = {

    label: stems

    for label, stems in noise_clusters.items()

    if len(set(stems)) >= MIN_CLUSTER_LEN
}

print("\n======================")
print("NOISE CLUSTERS")
print("======================\n")

for label, stems in noise_clusters.items():

    print(f"CLUSTER {label}")

    print(
        ", ".join(
            sorted(set(stems))
        )
    )

    print()

In [ ]:
# =====================================================
# BUILD STEM -> WORD FREQUENCY TABLE
# =====================================================

stem_word_frequencies = {}

for stem, occurrences in stem_occurrences.items():

    words = [

        occ["word"].lower()

        for occ in occurrences
    ]

    stem_word_frequencies[stem] = Counter(
        words
    )

# =====================================================
# CLUSTER NAMING FUNCTION
# =====================================================

def rename_clusters(
    cluster_dict,
    stem_word_frequencies
):

    renamed = {}

    for label, stems in cluster_dict.items():

        cluster_word_counter = Counter()

        for stem in stems:

            if stem in stem_word_frequencies:

                cluster_word_counter.update(
                    stem_word_frequencies[stem]
                )

        # Skip empty clusters

        if len(cluster_word_counter) == 0:
            continue

        # Highest frequency

        max_freq = max(
            cluster_word_counter.values()
        )

        # Words tied for highest frequency

        top_words = sorted([

            word

            for word, freq in
            cluster_word_counter.items()

            if freq == max_freq
        ])

        # Build cluster name

        cluster_name = " & ".join(
            top_words
        )

        renamed[cluster_name] = {

            "label": label,

            "stems": list(
                sorted(
                    set(stems)
                )
            ),

            "top_frequency": max_freq,

            "top_words": top_words
        }

    return renamed

# =====================================================
# RENAME ORIGINAL CLUSTERS
# =====================================================

renamed_clusters = rename_clusters(
    clusters,
    stem_word_frequencies
)

# =====================================================
# RENAME NOISE CLUSTERS
# =====================================================

renamed_noise_clusters = rename_clusters(
    noise_clusters,
    stem_word_frequencies
)

# =====================================================
# APPEND NOISE CLUSTERS TO MAIN CLUSTERS
# =====================================================

for cluster_name, cluster_data in (
    renamed_noise_clusters.items()
):

    final_name = cluster_name

    suffix = 2

    while final_name in renamed_clusters:

        final_name = (
            f"{cluster_name} ({suffix})"
        )

        suffix += 1

    renamed_clusters[
        final_name
    ] = cluster_data

# =====================================================
# PRINT FINAL CLUSTERS
# =====================================================

print("\n======================")
print("SEMANTIC CLUSTERS")
print("======================\n")

for cluster_name in sorted(
    renamed_clusters.keys()
):

    cluster_data = renamed_clusters[
        cluster_name
    ]

    print(
        f"CLUSTER: {cluster_name}"
    )


    print(
        "STEMS:"
    )

    print(
        ", ".join(
            cluster_data["stems"]
        )
    )

    print()

In [ ]:
# ------------------------------------------------------------
# Tokenize all sentences once
# ------------------------------------------------------------

def tokenize_many_for_analysis(
    texts,
    use_stopwords=True,
    use_lemmas=True,
):
    sentence_cache = {}

    for doc, text in zip(nlp.pipe(texts), texts):

        tokens = []

        for token in doc:

            if not token.is_alpha:
                continue

            if use_stopwords and token.is_stop:
                continue

            if use_lemmas:
                word = token.lemma_.lower()
            else:
                word = PorterStemmer().stem(token.text.lower())

            tokens.append(word)

        sentence_cache[text] = tokens

    return sentence_cache


# ------------------------------------------------------------
# Build sentence cache
# ------------------------------------------------------------

def build_sentence_token_cache(
    accepted_definitions,
    use_lemmas=True,
):

    sentences = set()

    for stem_data in accepted_definitions.values():

        for inst in stem_data.get("instances", []):

            sentence = inst.get("sentence", "").strip()

            if sentence:
                sentences.add(sentence)

    return tokenize_many_for_analysis(
        list(sentences),
        use_stopwords=True,
        use_lemmas=use_lemmas,
    )


# ------------------------------------------------------------
# Stem matching
# ------------------------------------------------------------

def normalize_cluster_stem(
    stem,
    use_lemmas,
):

    stem = stem.rstrip("*").lower()

    if use_lemmas:
        return stem

    return PorterStemmer().stem(stem)


def token_matches_cluster_stem(
    token,
    stems,
    use_lemmas,
):

    for stem in stems:

        normalized = normalize_cluster_stem(
            stem,
            use_lemmas,
        )

        if token == normalized:
            return True

        if token.startswith(normalized):
            return True

        if normalized.startswith(token):
            return True

    return False


# ------------------------------------------------------------
# Global n-gram statistics
# ------------------------------------------------------------

def build_global_ngram_statistics(
    sentence_cache,
    min_n=2,
    max_n=5,
):

    counter = Counter()

    for tokens in sentence_cache.values():

        for n in range(min_n, max_n + 1):

            if len(tokens) < n:
                continue

            for i in range(len(tokens) - n + 1):

                counter[tuple(tokens[i:i+n])] += 1

    frequencies = list(counter.values())

    threshold = (
        float(np.percentile(frequencies, 75))
        if frequencies
        else 0.0
    )

    print(f"Unique ngrams: {len(counter)}")
    print(f"75th percentile threshold: {threshold}")

    return counter, threshold


# ------------------------------------------------------------
# Assign ngrams to clusters
# ------------------------------------------------------------

def extract_cluster_ngrams(
    clusters,
    accepted_definitions,
    sentence_cache,
    global_ngram_counter,
    percentile_threshold,
    use_lemmas=True,
):

    cluster_ngrams = {}

    for cluster_id, stems in clusters.items():

        local_counter = Counter()

        for stem in stems:

            stem_data = accepted_definitions.get(stem, {})

            for inst in stem_data.get("instances", []):

                sentence = inst.get("sentence", "").strip()

                if sentence == "":
                    continue

                tokens = sentence_cache.get(sentence)

                if tokens is None:
                    continue

                for n in range(2, 6):

                    if len(tokens) < n:
                        continue

                    for i in range(len(tokens)-n+1):

                        gram = tuple(tokens[i:i+n])

                        if not any(
                            token_matches_cluster_stem(
                                token,
                                stems,
                                use_lemmas,
                            )
                            for token in gram
                        ):
                            continue

                        if global_ngram_counter.get(
                            gram,
                            0,
                        ) < percentile_threshold:
                            continue

                        local_counter[gram] += 1

        cluster_ngrams[cluster_id] = [
            " ".join(g)
            for g, _
            in local_counter.most_common()
        ]

    return cluster_ngrams

In [ ]:
sentence_cache = build_sentence_token_cache(
    accepted_definitions,
    use_lemmas=True
)

global_ngram_counter, percentile_threshold = \
    build_global_ngram_statistics(sentence_cache)

all_clusters = {
    **clusters,
    **noise_clusters
}

cluster_ngrams = extract_cluster_ngrams(
    all_clusters,
    accepted_definitions,
    sentence_cache,
    global_ngram_counter,
    percentile_threshold,
    use_lemmas=True,
)

# attach ngrams to renamed clusters

for cluster_name, cluster_data in renamed_clusters.items():

    label = cluster_data["label"]

    cluster_data["ngrams"] = cluster_ngrams.get(label, [])

In [ ]:
# Additionally, users may choose to rename clusters
# INTERACTIVE CLUSTER REVIEW

print("\n======================")
print("CLUSTER REVIEW")
print("======================\n")

final_clusters = {}

# Track excluded stems if the user chooses to exclude any
excluded_cluster_stems = set()

# Track excluded n-grams if the user chooses to exclude any
excluded_cluster_ngrams = set()

# SAVE ALL ORIGINAL STEMS
all_original_stems = set()

for cluster_data in renamed_clusters.values():

    all_original_stems.update(
        cluster_data["stems"]
    )


for cluster_name in sorted(
    renamed_clusters.keys()
):

    cluster_data = renamed_clusters[
        cluster_name
    ]

    stems = cluster_data["stems"]

    print("\n----------------------------------")
    print(f"CLUSTER NAME: {cluster_name}")
    if cluster_data.get("ngrams"):
        ngrams = cluster_data.get("ngrams", [])
        if ngrams:

            print("\nRepresentative n-grams:\n")

            for i, gram in enumerate(ngrams, 1):
                print(f"{i}. {gram}")

            remove_ngram_input = input("\nType n-gram numbers to remove "
                                       "(comma-separated) or press ENTER to keep all: ").strip()

            updated_ngrams = ngrams.copy()

            removed_ngrams = []

            if remove_ngram_input:

                try:

                    remove_indices = [
                        int(x.strip()) - 1 for x in remove_ngram_input.split(",")]

                    removed_ngrams = [gram for i, gram in enumerate(ngrams) if i in remove_indices
                                      ]

                    updated_ngrams = [gram for i, gram in enumerate(ngrams) if i not in remove_indices]

                    excluded_cluster_ngrams.update(removed_ngrams)

                except:
                    print("Invalid n-gram selection.")
            

    print("\nSTEMS:")

    for i, stem in enumerate(stems):

        print(f"{i+1}. {stem}")

    print("\nAccept this cluster?")

    accept = input(
        "(y = accept / n = modify): "
    ).strip().lower()


    # ACCEPT AS-IS

    if accept == "y":

        final_clusters[cluster_name] = {
            "stems": stems,
            "ngrams": updated_ngrams,
            "excluded_ngrams": removed_ngrams,
            "excluded_stems": []
}

        continue

    # CHANGE CLUSTER NAME

    new_name = input(
        "\nNew cluster name "
        "(leave empty to keep current): "
    ).strip()

    if new_name == "":

        new_name = cluster_name


    # REMOVE STEMS

    print("\nCurrent stems:")

    for i, stem in enumerate(stems):

        print(f"{i+1}. {stem}")

    remove_input = input(
        "\nType stem numbers to remove "
        "\n(comma-separated and in any order as in '5,2,3,9...') "
        "\n or press ENTER to keep all: "
    ).strip()

    updated_stems = stems.copy()

    removed_stems = []

    if remove_input != "":

        try:

            remove_indices = [

                int(x.strip()) - 1

                for x in remove_input.split(",")
            ]

            removed_stems = [

                stem

                for i, stem in enumerate(stems)

                if i in remove_indices
            ]

            updated_stems = [

                stem

                for i, stem in enumerate(stems)

                if i not in remove_indices
            ]

            # ASK WHICH REMOVED STEMS SHOULD
            # ALSO BE GLOBALLY EXCLUDED

            if len(removed_stems) > 0:

                print("\nRemoved stems:")

                for i, stem in enumerate(removed_stems):

                    print(f"{i+1}. {stem}")

                exclude_input = input(
                    "\nWhich removed stems should "
                    "also be added to excList?\n"
                    "(comma-separated numbers "
                    "or ENTER for none): ").strip()

                if exclude_input != "":

                    try:

                        exclude_indices = [
                            int(x.strip()) - 1 for x in exclude_input.split(",")]

                        globally_excluded = [stem for i, stem in enumerate(removed_stems) if i in exclude_indices]

                        excluded_cluster_stems.update(globally_excluded)

                    except:
                        print("\nInvalid exclusion input.")
                        print("No stems added to excList.")

        except:

            print(
                "\nInvalid input."
            )

            print(
                "Keeping all stems."
            )

    # SAVE UPDATED CLUSTER

    final_clusters[new_name] = {
        "stems": updated_stems,
        "ngrams": cluster_data.get("ngrams", []),
        "excluded_ngrams": removed_ngrams,
        "excluded_stems": removed_stems
}

# FINAL OUTPUT

print("\n======================")
print("FINAL CLUSTERS")
print("======================\n")

for cluster_name in sorted(
    final_clusters.keys()
):

    print(
        f"CLUSTER: {cluster_name}"
    )

    print(
        "STEMS:"
    )

    print(
        ", ".join(
            final_clusters[
                cluster_name
            ]["stems"]
        )
    )

    ngrams = final_clusters[cluster_name].get("ngrams", [])
    if ngrams:
        print("\nN-GRAMS:")
        print(", ".join(ngrams[:100]))

    excluded_ngrams = final_clusters[
        cluster_name
        ].get("excluded_ngrams", [])

    if excluded_ngrams:

        print("\nEXCLUDED N-GRAMS:")

        print(", ".join(excluded_ngrams))

    excluded = final_clusters[
        cluster_name
    ]["excluded_stems"]

    if len(excluded) > 0:

        print(
            "EXCLUDED STEMS:"
        )

        print(
            ", ".join(excluded)
        )

    print()

In [ ]:
print("\n======================")
print("VOYANT CORPUS")
print("======================\n")

corpus_id = input(
    "Enter Voyant Corpus ID: "
).strip()

use_smart_stopwords = input(
    "\nUse Voyant en_smart stopwords? (y/n): "
).strip().lower() == "y"

smart_stopwords = []

if use_smart_stopwords:

    with open(
        "stop.en.smart.txt",
        "r",
        encoding="utf-8"
    ) as f:

        smart_stopwords = [

            line.strip()

            for line in f

            if line.strip()
        ]

In [ ]:
# --------------------------------------------------
# SAVE JSON
# --------------------------------------------------

full_exc_list = set(excluded_cluster_stems)

if use_smart_stopwords:
    full_exc_list.update(smart_stopwords)

# Optional: build a global list of excluded n-grams
full_excluded_ngrams = sorted(
    list(excluded_cluster_ngrams)
) if "excluded_cluster_ngrams" in globals() else []

phase1_state = {

    # Corpus ID for Voyant integration
    "corpusId": corpus_id,

    # Included stems
    "incList": sorted(
        list(
            all_original_stems
            - excluded_cluster_stems
        )
    ),

    # Excluded stems
    "excList": sorted(full_exc_list),

    # Globally excluded n-grams
    "excludedNgrams": full_excluded_ngrams,

    # Cluster definitions
    "clusterDefs": [

        {

            # Cluster name
            "name": cluster_name,

            # Final stems
            "stems": final_clusters[
                cluster_name
            ]["stems"],

            # Representative n-grams
            "ngrams": final_clusters[
                cluster_name
            ].get(
                "ngrams",
                []
            ),

            # Removed stems
            "excluded_stems": final_clusters[
                cluster_name
            ].get(
                "excluded_stems",
                []
            ),

            # Removed n-grams
            "excluded_ngrams": final_clusters[
                cluster_name
            ].get(
                "excluded_ngrams",
                []
            )

        }

        for cluster_name in sorted(
            final_clusters.keys()
        )
    ]
}

# --------------------------------------------------
# SAVE JSON
# --------------------------------------------------

with open(
    FINAL_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        phase1_state,
        f,
        indent=4,
        ensure_ascii=False
    )

print(f"\nSaved JSON to {FINAL_JSON}")

In [ ]:
# HTML CLUSTER EXPORT

TABLEAU20 = [
    "#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F",
    "#EDC948","#B07AA1","#FF9DA7","#9C755F","#BAB0AC",
    "#499894","#A0CBE8","#FFBE7D","#FF9D9A","#86BCB6",
    "#8CD17D","#F1CE63","#D4A6C8","#FABFD2","#D7B5A6",
]


# --------------------------------------------------
# HEX TO RGB
# --------------------------------------------------

def hex_to_rgb(h):

    h = h.lstrip("#")

    return tuple(
        int(h[i:i+2], 16)
        for i in (0, 2, 4)
    )


# --------------------------------------------------
# BUILD TABLE ROWS
# --------------------------------------------------

rows = []

for i, cluster_name in enumerate(

    sorted(final_clusters.keys())

):

    cluster_data = final_clusters[
        cluster_name
    ]

    stems = cluster_data.get(
        "stems",
        []
    )

    excluded_stems = cluster_data.get(
        "excluded_stems",
        []
    )

    ngrams = cluster_data.get(
        "ngrams",
        []
    )

    excluded_ngrams = cluster_data.get(
        "excluded_ngrams",
        []
    )

    color = TABLEAU20[
        i % len(TABLEAU20)
    ]

    r, g, b = hex_to_rgb(color)

    # --------------------------------------------------
    # INCLUDED STEMS
    # --------------------------------------------------

    stems_html = ", ".join(

        f"<code>{stem}</code>"

        for stem in stems
    )

    # --------------------------------------------------
    # INCLUDED NGRAMS
    # --------------------------------------------------

    ngrams_html = ""

    if len(ngrams) > 0:

        ngrams_str = ", ".join(

            f"<code>{gram}</code>"

            for gram in ngrams
        )

        ngrams_html = (

            f'<div style="margin-top:8px;">'
            f'<strong>N-grams:</strong> '
            f'{ngrams_str}'
            f'</div>'

        )

    # --------------------------------------------------
    # EXCLUDED STEMS
    # --------------------------------------------------

    excluded_stems_html = ""

    if len(excluded_stems) > 0:

        excluded_str = ", ".join(

            f"<code>{stem}</code>"

            for stem in excluded_stems
        )

        excluded_stems_html = (

            f'<div style="margin-top:6px;'
            f'font-size:0.80em;'
            f'color:#999;">'
            f'<strong>Excluded stems:</strong> '
            f'{excluded_str}'
            f'</div>'

        )

    # --------------------------------------------------
    # EXCLUDED NGRAMS
    # --------------------------------------------------

    excluded_ngrams_html = ""

    if len(excluded_ngrams) > 0:

        excluded_ngrams_str = ", ".join(

            f"<code>{gram}</code>"

            for gram in excluded_ngrams
        )

        excluded_ngrams_html = (

            f'<div style="margin-top:4px;'
            f'font-size:0.80em;'
            f'color:#999;">'
            f'<strong>Excluded n-grams:</strong> '
            f'{excluded_ngrams_str}'
            f'</div>'

        )

    # --------------------------------------------------
    # BUILD TABLE ROW
    # --------------------------------------------------

    rows.append(

        f'    <tr>\n'

        f'      <td style="padding:5px 12px 5px 0;">'
        f'&nbsp;</td>\n'

        f'      <td style="padding:5px 12px 5px 0;'
        f'color:rgb({r},{g},{b});'
        f'font-weight:bold;'
        f'vertical-align:top;">'
        f'{cluster_name}'
        f'</td>\n'

        f'      <td style="padding:5px 0;'
        f'font-size:0.88em;'
        f'color:#555;">'

        f'<strong>Stems:</strong><br>'
        f'{stems_html}'

        f'{ngrams_html}'

        f'{excluded_stems_html}'

        f'{excluded_ngrams_html}'

        f'</td>\n'

        f'    </tr>'

    )


# --------------------------------------------------
# JOIN ROWS
# --------------------------------------------------

rows_html = "\n".join(rows)


# --------------------------------------------------
# SUMMARY STATISTICS
# --------------------------------------------------

total_stems = sum(

    len(cluster.get("stems", []))

    for cluster in final_clusters.values()

)

total_ngrams = sum(

    len(cluster.get("ngrams", []))

    for cluster in final_clusters.values()

)


# --------------------------------------------------
# HTML
# --------------------------------------------------

snippet = f"""
<h3>Semantic Clusters — Phase 1 Results</h3>

<p style="font-style:italic;
          color:#666;
          font-size:0.9em;">

  {corpus_name} &mdash;
  {len(final_clusters)} clusters &middot;
  {total_stems} stems &middot;
  {total_ngrams} n-grams &middot;
  Tableau20 palette

</p>

<table style="border-collapse:collapse;
              font-family:serif;
              font-size:14px;">

  <thead>

    <tr>

      <th style="padding:5px 12px 5px 0;">
        &nbsp;
      </th>

      <th style="padding:5px 12px 5px 0;
                 text-align:left;">
        Cluster
      </th>

      <th style="padding:5px 0;
                 text-align:left;">
        Contents
      </th>

    </tr>

  </thead>

  <tbody>

{rows_html}

  </tbody>

</table>
"""


# --------------------------------------------------
# SAVE HTML
# --------------------------------------------------

with open(
    HTML_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(snippet)

print(
    f"\nHTML cluster snippet written:\n"
    f"{HTML_PATH}"
)